<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 02 — Custom Tools

In [01](01_providers_and_agents.ipynb) you ran agents with the **built-in** tool pool (`read_file`, `list_dir`, `search`, …). This notebook shows how to give an agent *new* capabilities by writing your own tools.

A **tool** is a small, typed unit of work the model can call by name:

```
model decides → emits a tool call (name + JSON args)
            ↓
agent validates args against your schema → runs your code → feeds the result back
```

You'll learn the whole tool contract:

1. **Anatomy of a `Tool`** — name, description, a Pydantic `input_model`, and an async `call`
2. **Your first tool** — define it, inspect its schema, run it directly
3. **Input validation** — bad arguments come back as a *tool error*, never a crash
4. **Run it in an `AgenticLoop`** — build a custom `ToolPool` and let the model use your tool
5. **`ToolContext`** — reach the working directory, the user, memory, and more
6. **Behaviour flags** — `is_read_only` / `is_concurrency_safe` / `is_destructive`
7. **Register globally** — surface your tool in `default_pool()` (and the CLI) via a factory
8. **`ReactAgent`** — the same custom tool with a text-parsing agent

**Model used:** `google/gemma-4-12b-qat` served via LM Studio on `localhost:1234`

---

## Contents
1. [Setup](#1-setup)
2. [Anatomy of a Tool](#2-anatomy-of-a-tool)
3. [Your first custom tool](#3-your-first-custom-tool)
4. [Input validation](#4-input-validation)
5. [Run it in an AgenticLoop](#5-run-it-in-an-agenticloop)
6. [ToolContext — cwd, user, memory](#6-toolcontext--cwd-user-memory)
7. [Behaviour flags & permissions](#7-behaviour-flags--permissions)
8. [Register globally](#8-register-globally)
9. [Custom tools with ReactAgent](#9-custom-tools-with-reactagent)


---

## 1. Setup

Same boilerplate as notebook 01: point Python at the repo root, configure the LM Studio
connection, and define a notebook-friendly `IOHandler`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

version,0.2.0 | python 3.11.15
os,Linux 6.17.0-35-generic (x86_64)
torch,2.11.0+cu128 CUDA: yes (12.8)
mps,no (built: False)
transformers,5.2.0 sentEmb 5.5.0
accelerate,1.13.0 bnb -
gpu,NVIDIA GeForce RTX 5080


neurosurfer 0.2.0


In [3]:
# ── LM Studio connection ──────────────────────────────────────
# Make sure LM Studio is running with the model loaded and the local server ON.
# Server → default port 1234.  Copy the model identifier from the LM Studio UI.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "google/gemma-4-12b-qat"   # adjust if your name differs
CONTEXT_WINDOW  = 32_768                       # check your model's context length

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",   # LM Studio ignores this; any non-empty string works
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")
print(f"Native tool calling: {provider.capabilities.tool_call_style}")

Provider: google/gemma-4-12b-qat
Native tool calling: openai


In [4]:
# ── JupyterIO ───────────────────────────────────────────────
# Agents need an IOHandler for interactive prompts (plan approval, shell commands,
# out-of-scope writes).  This notebook-friendly version auto-approves everything
# and prints notifications so you can see what's happening.

class JupyterIO:
    """Auto-approving IOHandler for notebook demos."""

    async def ask(self, question: str, options=None) -> str:
        print(f"  [ask] {question}")
        return "yes"

    async def request_plan_approval(self, plan: str) -> tuple[bool, str]:
        print(f"  [plan approved]")
        return True, ""

    async def request_shell_approval(self, command: str, reason: str) -> bool:
        print(f"  [shell approved] {command}")
        return True

    async def request_write_approval(self, path: str, summary: str) -> str:
        print(f"  [write approved] {path}")
        return "once"

    def notify(self, message: str) -> None:
        print(f"  [notice] {message}")

print("JupyterIO ready.")

JupyterIO ready.


---

## 2. Anatomy of a Tool

Every tool subclasses `neurosurfer.tools.Tool` and supplies four things:

| Attribute | Type | Purpose |
|---|---|---|
| `name` | `str` | Unique identifier the model calls (snake_case) |
| `description` | `str` | What it does + *when to use it* — this is the model's only guide |
| `input_model` | `type[BaseModel]` | A **Pydantic** model describing the arguments |
| `call(self, args, ctx)` | `async` method | Your logic; returns a `ToolResult` |

Two rules make tools robust inside an agent loop:

- **Errors are *returned*, not raised.** Return `ToolResult.error("…")` (or just raise — the
  base class catches it). The agent feeds the error back to the model so it can self-correct,
  instead of the whole run crashing.
- **The schema is generated from `input_model`.** The field types and `Field(description=...)`
  become the JSON schema sent to the model — so good field descriptions = good tool calls.

```python
from neurosurfer.tools import Tool, ToolResult, ToolContext

class MyTool(Tool):
    name = "my_tool"
    description = "What it does. When to use it."
    input_model = MyArgs            # a pydantic BaseModel

    async def call(self, args: MyArgs, ctx: ToolContext) -> ToolResult:
        ...
        return ToolResult.ok("the result text")
```

`ToolResult` carries `content: str` and `is_error: bool`. Use the helpers
`ToolResult.ok(text)` and `ToolResult.error(text)`.

---

## 3. Your first custom tool

A `word_count` tool: it takes some text and returns word/character counts. Deterministic,
no external dependencies, and **read-only** (it changes nothing) — the simplest useful shape.

In [5]:
from pydantic import BaseModel, Field
from neurosurfer.tools import Tool, ToolResult, ToolContext


class WordCountArgs(BaseModel):
    text: str = Field(description="The text to analyze.")


class WordCountTool(Tool):
    name = "word_count"
    description = (
        "Count the words and characters in a piece of text. "
        "Use when the user asks how long a passage is."
    )
    input_model = WordCountArgs

    # Read-only → the loop may run it without write/shell approval and in parallel.
    def is_read_only(self, args: BaseModel) -> bool:
        return True

    # Friendly status line shown by front-ends while the tool runs.
    def progress_message(self, args: dict) -> str:
        return "Counting words…"

    async def call(self, args: WordCountArgs, ctx: ToolContext) -> ToolResult:
        words = len(args.text.split())
        chars = len(args.text)
        return ToolResult.ok(f"words={words}, characters={chars}")


print("WordCountTool defined.")

WordCountTool defined.


The schema the model sees is generated automatically from `WordCountArgs`. Let's inspect it,
then run the tool **directly** (no agent) — every tool exposes an async `run(raw_args, ctx)`
that validates the arguments and dispatches to `call`.

In [6]:
import asyncio, json
from pathlib import Path

tool = WordCountTool()

# 1. The schema sent to the model — note the field description carries through.
print("Schema:")
print(json.dumps(tool.schema.input_schema, indent=2))

# 2. Run it directly. Tools need a ToolContext (cwd + io at minimum).
ctx = ToolContext(cwd=repo_root, io=JupyterIO())

result = await tool.run({"text": "Neurosurfer makes custom tools easy"}, ctx)
print("\nresult.content :", result.content)
print("result.is_error:", result.is_error)

Schema:
{
  "properties": {
    "text": {
      "description": "The text to analyze.",
      "type": "string"
    }
  },
  "required": [
    "text"
  ],
  "type": "object",
  "additionalProperties": false
}

result.content : words=5, characters=35
result.is_error: False


---

## 4. Input validation

`run()` validates the raw arguments against `input_model` **before** calling your code.
If the model emits a malformed call (missing field, wrong type), you get a
`ToolResult` with `is_error=True` — not an exception. The agent loop appends that error
to the conversation so the model can fix its next call.

In [7]:
# Missing the required 'text' field
bad = await tool.run({}, ctx)
print("missing field → is_error =", bad.is_error)
print(bad.content)

print()

# Wrong type is coerced or rejected by pydantic the same way
bad2 = await tool.run({"text": 12345}, ctx)   # int, not str
print("wrong type  → is_error =", bad2.is_error, "|", bad2.content)

missing field → is_error = True
Invalid arguments for word_count: 1 validation error for WordCountArgs
text
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

wrong type  → is_error = True | Invalid arguments for word_count: 1 validation error for WordCountArgs
text
  Input should be a valid string [type=string_type, input_value=12345, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


---

## 5. Run it in an AgenticLoop

Tools are handed to an agent as a **`ToolPool`**. You can build one from scratch, or extend
the built-ins. Here we give the agent a small, focused pool: our `word_count` tool plus the
`finish` tool (which every multi-step agent needs to end the run).

> A custom-arithmetic tool is the classic example of *why* tools matter: LLMs are unreliable at
> exact math. Below we also add a `unit_convert` tool and let the model delegate the calculation.

In [8]:
class UnitConvertArgs(BaseModel):
    value: float = Field(description="The numeric value to convert.")
    from_unit: str = Field(description="Source unit: km, mi, kg, lb, c, f.")
    to_unit: str = Field(description="Target unit: km, mi, kg, lb, c, f.")


class UnitConvertTool(Tool):
    name = "unit_convert"
    description = (
        "Convert a value between units (km↔mi, kg↔lb, c↔f). "
        "LLMs are unreliable at exact arithmetic — always use this tool for conversions."
    )
    input_model = UnitConvertArgs

    _FACTORS = {
        ("km", "mi"): lambda v: v * 0.621371,
        ("mi", "km"): lambda v: v / 0.621371,
        ("kg", "lb"): lambda v: v * 2.20462,
        ("lb", "kg"): lambda v: v / 2.20462,
        ("c", "f"):   lambda v: v * 9 / 5 + 32,
        ("f", "c"):   lambda v: (v - 32) * 5 / 9,
    }

    def is_read_only(self, args: BaseModel) -> bool:
        return True

    async def call(self, args: UnitConvertArgs, ctx: ToolContext) -> ToolResult:
        key = (args.from_unit.lower(), args.to_unit.lower())
        fn = self._FACTORS.get(key)
        if fn is None:
            supported = sorted({u for k in self._FACTORS for u in k})
            return ToolResult.error(
                f"Cannot convert {args.from_unit}→{args.to_unit}. Supported units: {supported}"
            )
        return ToolResult.ok(f"{args.value} {args.from_unit} = {fn(args.value):.4f} {args.to_unit}")


print("UnitConvertTool defined.")

UnitConvertTool defined.


In [9]:
from neurosurfer.tools import ToolPool
from neurosurfer.tools.builtin import FinishTool

# A pool containing ONLY our custom tools + finish.
custom_pool = ToolPool([WordCountTool(), UnitConvertTool(), FinishTool()])
print("Custom pool:", [t.name for t in custom_pool.all()])

Custom pool: ['word_count', 'unit_convert', 'finish']


In [10]:
from neurosurfer.agents import AgenticLoop, Guardrails, events

# verbose=False here because this cell renders every event by hand (see §5b for the
# zero-boilerplate version). With the default verbose=True the trace would log twice.
tool_agent = AgenticLoop(
    provider      = provider,
    tools         = custom_pool,
    system_prompt = (
        "You are a helpful assistant. You have a unit_convert tool and a word_count tool. "
        "Always use unit_convert for any unit conversion instead of computing it yourself. "
        "Call the finish tool when the task is complete."
    ),
    guardrails    = Guardrails(max_turns=8),
    io            = JupyterIO(),
    cwd           = repo_root,
    verbose       = False,
)

task = "How many miles is 42 kilometers? And how many words are in the sentence 'the quick brown fox jumps'?"
print(f"Task: {task}\n{'─' * 60}")

async for ev in tool_agent.run(task):
    if isinstance(ev, events.TextDelta):          # ← TextDelta = the streamed answer
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.ToolStarted):
        print(f"\n\n  [→ tool] {ev.name}({ev.args})")
    elif isinstance(ev, events.ToolFinished):
        print(f"  [← result] {ev.result.content}")
    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}\nRun finished — status: {ev.status}")

Task: How many miles is 42 kilometers? And how many words are in the sentence 'the quick brown fox jumps'?
────────────────────────────────────────────────────────────


  [→ tool] unit_convert({'from_unit': 'km', 'to_unit': 'mi', 'value': 42})
  [← result] 42.0 km = 26.0976 mi


  [→ tool] word_count({'text': 'the quick brown fox jumps'})
  [← result] words=5, characters=25
42 kilometers is approximately 26.1 miles, and there are 5 words in the sentence "the quick brown fox jumps".



  [→ tool] finish({'status': 'success', 'summary': 'Converted 42 kilometers to miles and counted the words in the provided sentence.'})
  [← result] Converted 42 kilometers to miles and counted the words in the provided sentence.


────────────────────────────────────────────────────────────
Run finished — status: success


### 5b. You don't have to render every event

The loop above is the *full* event API — useful when you're building a UI. But for everyday
use it's boilerplate. Every agent runs with **`verbose=True`** by default, which shows a **live
animated status** as a side-channel — a spinner whose label shifts (*Thinking… → Pondering… →
Cooking…*) while the model reasons, *Running &lt;tool&gt;…* while a tool executes, and a line per
completed tool call — *independent of which events you consume*. (In a plain terminal/CI it
degrades to static `→ tool` / `↳ result` lines; pass `verbose=False` to silence it.)

So the minimal version below gets that live trace for free while you handle only the one thing
you actually care about: **`TextDelta`**, the streamed answer. No `thinking` flag, no tool-event
plumbing.

In [ ]:
minimal_agent = AgenticLoop(
    provider      = provider,
    tools         = custom_pool,
    system_prompt = "Use unit_convert for conversions. Call finish when done.",
    guardrails    = Guardrails(max_turns=8),
    io            = JupyterIO(),
    cwd           = repo_root,
    # verbose=True is the default — the activity trace prints itself.
)

# The live spinner (Thinking…/Running unit_convert…) shows automatically (verbose);
# all you render is the streamed answer.
async for ev in minimal_agent.run("Convert 5 kg to pounds."):
    if isinstance(ev, events.TextDelta):
        print(ev.text, end="", flush=True)

→ unit_convert({'from_unit': 'kg', 'to_unit': 'lb', 'value': 5})
  ↳ 5.0 kg = 11.0231 lbm
           king…
5
 kg
 is
 equal
 to
 
1
1
.
0
2
3
1
 pounds
.


---

## 6. ToolContext — cwd, user, memory

The second argument to `call` is a **`ToolContext`** — everything the tool needs that isn't in
its own arguments. The most useful fields:

| Field | What it gives you |
|---|---|
| `ctx.cwd` | The agent's working directory (`Path`) — resolve relative paths against it |
| `ctx.io` | The `IOHandler` — `await ctx.io.ask(...)`, `ctx.io.notify(...)` |
| `ctx.memory` | Long-term `MemoryStore` (None when memory is disabled) — covered in nb 03 |
| `ctx.durable` | Per-run scratch state shared across tool calls |
| `ctx.spawn` | Spawn a sub-agent: `await ctx.spawn(agent_type, prompt)` |

Here's a `dir_stats` tool that uses `ctx.cwd` to resolve a path and report directory size.

In [ ]:
class DirStatsArgs(BaseModel):
    subdir: str = Field(default=".", description="Directory (relative to the working dir) to inspect.")


class DirStatsTool(Tool):
    name = "dir_stats"
    description = "Report how many files a directory contains and their total size in KB."
    input_model = DirStatsArgs

    def is_read_only(self, args: BaseModel) -> bool:
        return True

    async def call(self, args: DirStatsArgs, ctx: ToolContext) -> ToolResult:
        target = (ctx.cwd / args.subdir).resolve()          # ← resolve against the agent's cwd
        if not target.is_dir():
            return ToolResult.error(f"Not a directory: {args.subdir}")
        files = [p for p in target.rglob("*") if p.is_file()]
        total_kb = sum(p.stat().st_size for p in files) / 1024
        ctx.io.notify(f"scanned {target.name}")             # ← talk to the user via ctx.io
        return ToolResult.ok(f"{target.name}: {len(files)} files, {total_kb:.1f} KB total")


# Run it directly against the repo's tools/ folder
ctx = ToolContext(cwd=repo_root, io=JupyterIO())
res = await DirStatsTool().run({"subdir": "neurosurfer/tools"}, ctx)
print(res.content)

---

## 7. Behaviour flags & permissions

Three methods tell the agent loop *how* a tool behaves. They drive scheduling (can it run in
parallel?) and gating (does it need approval?):

| Method | Default | Meaning |
|---|---|---|
| `is_read_only(args)` | `False` | Doesn't modify anything → no write approval, eligible for parallel runs |
| `is_concurrency_safe(args)` | `= is_read_only` | Safe to run alongside other tools in the same turn |
| `is_destructive(args)` | `= not read_only` | Irreversible/dangerous → the loop is extra careful |

They take `args`, so a tool can decide *per call*. A `run_command` tool, for example, is read-only
for `ls` but destructive for `rm`:

```python
def is_read_only(self, args) -> bool:
    return args.command.split()[0] in {"ls", "cat", "grep", "find"}
```

**Rule of thumb:** if your tool only reads, override `is_read_only` to return `True` (as both tools
above do). If it writes files, runs shell, or calls a mutating API, leave the defaults — the
`Guardrails` + `IOHandler` will then gate it through `request_write_approval` /
`request_shell_approval`.

---

## 8. Register globally

Passing a hand-built `ToolPool` is perfect for notebooks and scripts. But to make a tool appear
**everywhere** — in `default_pool()`, in agents that use the full pool, and in the `neurosurfer`
CLI — register a **factory** (a zero-arg callable returning a fresh instance):

```python
from neurosurfer.tools.registry import register_tool_factory
register_tool_factory(lambda: UnitConvertTool())
```

Do this once at import time (e.g. in your package's `__init__.py`). It's idempotent per factory,
so re-imports won't duplicate it. A built-in always wins a name clash.

In [ ]:
from neurosurfer.tools.registry import register_tool_factory
from neurosurfer.tools import default_pool

register_tool_factory(lambda: UnitConvertTool())

pool = default_pool()
print("default_pool now includes unit_convert:", pool.get("unit_convert") is not None)
print("total tools in default pool:", len(pool.names()))

---

## 9. Custom tools with ReactAgent

Custom tools are agent-agnostic — the *same* `Tool` class works with `ReactAgent`, which drives
models that lack native function calling by parsing `Thought / Action / Action Input` text. Your
tool's `name`, `description`, and schema are rendered into the ReAct prompt automatically.

`ReactAgent` shares the **exact same** live trace as `AgenticLoop` (it lives in the base agent),
so the minimal consumer is identical: stream `TextDelta` (the answer) and let `verbose=True` show
the animated Thinking…/Running… status for us. Note the final answer **streams** token by token —
`ReactAgent` emits it incrementally as `TextDelta` the moment it parses `Final Answer:`.

In [ ]:
from neurosurfer.agents import ReactAgent

react_agent = ReactAgent(
    provider      = provider,
    tools         = ToolPool([UnitConvertTool(), FinishTool()]),
    system_prompt = "You are a precise assistant. Use unit_convert for any conversion. Think step by step.",
    guardrails    = Guardrails(max_turns=8),
    io            = JupyterIO(),
    cwd           = repo_root,
    # verbose=True (default) → the reasoning/tool trace logs itself.
)

task = "Convert 100 degrees Celsius to Fahrenheit, then tell me if that's hot."
print(f"Task: {task}\n{'─' * 60}")

# The live spinner shows itself (verbose); you render only the answer.
async for ev in react_agent.run(task):
    if isinstance(ev, events.TextDelta):     # the streamed Final Answer — yours to render
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.RunFinished): # ev.report also holds the full answer in one piece
        print(f"\n\n{'─' * 60}\nRun finished — {ev.status}")

---

## Summary

You now know the full tool contract:

| Step | API |
|---|---|
| Define arguments | a Pydantic `BaseModel` with `Field(description=...)` |
| Define the tool | subclass `Tool`: `name`, `description`, `input_model`, `async call` |
| Return a result | `ToolResult.ok(text)` / `ToolResult.error(text)` |
| Declare behaviour | `is_read_only` / `is_concurrency_safe` / `is_destructive` |
| Reach the environment | `ctx.cwd`, `ctx.io`, `ctx.memory`, `ctx.durable`, `ctx.spawn` |
| Use with an agent | wrap in a `ToolPool` and pass `tools=...` |
| Register everywhere | `register_tool_factory(lambda: MyTool())` |

**Key ideas**
- Errors are *returned* (`ToolResult.error`), never raised — the agent self-corrects.
- The model only sees `name` + `description` + the generated schema. Write them for the model.
- The same `Tool` works with `Agent`, `AgenticLoop`, and `ReactAgent`.
- **`TextDelta` is the streamed answer; `ThinkingDelta` is reasoning.** You normally only render
  `TextDelta` — with `verbose=True` (default), a live animated status (Thinking…/Running &lt;tool&gt;…)
  shows itself, so you never hand-roll an event ladder just to see what the agent is doing. The
  trace lives in the base agent, so `AgenticLoop` and `ReactAgent` behave identically.

---

## What's Next

| # | Notebook | Topic |
|---|---|---|
| 01 | *Providers, Agents & RAG* | Core building blocks |
| **02** | *Custom Tools* ← **you are here** | Define and register your own tools |
| 03 | Memory & Sessions | Persistent agent memory across conversations |
| 04 | The Gateway Server | Serve agents as OpenAI-compatible endpoints |
| 05 | Workflow Builder | Describe → design → build a multi-step DAG |
